In [8]:
%cd Proyecto_Big_Data

/home/jovyan/work/Proyecto_Big_Data


In [9]:
import requests
import pandas as pd
import time
from datetime import datetime

# --- CONFIGURACIÓN ---
CSV_DEMANDA = "demanda_electrica_2013_2026.csv"
BASE_URL = "https://apidatos.ree.es/es/datos/demanda/evolucion"

def descargar_historico_demanda(anio_inicio=2013):
    ahora = pd.Timestamp.now()
    lista_demanda = []
    
    for anio in range(anio_inicio, ahora.year + 1):
        # Lógica de fechas
        start_m = "05" if anio == 2013 else "01"
        end_date = ahora.strftime('%Y-%m-%d') if anio == ahora.year else f"{anio}-12-31"
        url = f"{BASE_URL}?start_date={anio}-{start_m}-01T00:00&end_date={end_date}T23:59&time_trunc=day"
        
        # --- BUCLE DE REINTENTOS PARA CADA AÑO ---
        exito = False
        reintentos = 3
        while not exito and reintentos > 0:
            try:
                print(f"📥 Solicitando año {anio}...", end=" ")
                res = requests.get(url, timeout=40) # Subimos a 40 segundos por si acaso
                
                if res.status_code == 200:
                    data = res.json()
                    valores = data['included'][0]['attributes']['values']
                    df_year = pd.DataFrame(valores)
                    df_year = df_year[['datetime', 'value']]
                    lista_demanda.append(df_year)
                    print(f"✅ ({len(df_year)} días)")
                    exito = True # Salimos del bucle while
                else:
                    print(f"⚠️ Error {res.status_code}. Reintentando...")
                    reintentos -= 1
                    time.sleep(5) # Esperamos 5 segundos si el servidor falla
            
            except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
                print("⏳ Timeout/Red... Reintentando en 5 segundos...")
                reintentos -= 1
                time.sleep(5)
            except Exception as e:
                print(f"💥 Error inesperado: {e}")
                break # Errores graves paran el bucle del año

        time.sleep(1) # Respiro entre años con éxito

    if lista_demanda:
        df_final = pd.concat(lista_demanda, ignore_index=True)
        # Limpieza blindada de fechas
        df_final['datetime'] = pd.to_datetime(df_final['datetime'], utc=True)
        df_final['fecha'] = df_final['datetime'].dt.tz_convert('Europe/Madrid').dt.tz_localize(None).dt.normalize()
        
        df_final = df_final[['fecha', 'value']].rename(columns={'value': 'demanda_mw'})
        df_final = df_final.drop_duplicates(subset=['fecha']).sort_values('fecha')
        
        df_final.to_csv(CSV_DEMANDA, index=False)
        print(f"\n🚀 PROCESO FINALIZADO: {len(df_final)} registros guardados en {CSV_DEMANDA}")
        return df_final

# Ejecutar
df_demanda = descargar_historico_demanda()

📥 Solicitando año 2013... ✅ (245 días)
📥 Solicitando año 2014... ✅ (365 días)
📥 Solicitando año 2015... ✅ (365 días)
📥 Solicitando año 2016... ✅ (366 días)
📥 Solicitando año 2017... ✅ (365 días)
📥 Solicitando año 2018... ✅ (365 días)
📥 Solicitando año 2019... ✅ (365 días)
📥 Solicitando año 2020... ✅ (366 días)
📥 Solicitando año 2021... ✅ (365 días)
📥 Solicitando año 2022... ✅ (365 días)
📥 Solicitando año 2023... ✅ (365 días)
📥 Solicitando año 2024... ✅ (366 días)
📥 Solicitando año 2025... ✅ (365 días)
📥 Solicitando año 2026... ✅ (113 días)

🚀 PROCESO FINALIZADO: 4741 registros guardados en demanda_electrica_2013_2026.csv


In [10]:
df = pd.read_csv("demanda_electrica_2013_2026.csv")

display(df.head(10000))

,fecha,demanda_mw
0,2013-05-01,588773.328
1,2013-05-02,666701.010
2,2013-05-03,682986.809
3,2013-05-04,616333.383
4,2013-05-05,565184.363
...,...,...
4736,2026-04-19,536183.933
4737,2026-04-20,645687.583
4738,2026-04-21,661719.332
4739,2026-04-22,648840.800


In [11]:
import pandas as pd

# 1. CARGAR LOS DATASETS
print("📖 Cargando archivos...")
df_clima_fest = pd.read_csv("clima_festivos_consolidado.csv")
df_demanda = pd.read_csv("demanda_electrica_2013_2026.csv")

# 2. NORMALIZAR FECHAS (Paso crítico para que coincidan)
# Aseguramos que ambas columnas sean datetime y solo tengan YYYY-MM-DD
df_clima_fest['fecha'] = pd.to_datetime(df_clima_fest['fecha']).dt.normalize()
df_demanda['fecha'] = pd.to_datetime(df_demanda['fecha']).dt.normalize()

print(f"📊 Registros clima: {len(df_clima_fest)}")
print(f"📊 Registros demanda: {len(df_demanda)}")

# 3. UNIÓN MAESTRA (MERGE)
# Usamos 'inner' para asegurar que cada fila tenga información completa
df_final = pd.merge(df_clima_fest, df_demanda, on='fecha', how='inner')

# 4. LIMPIEZA Y ORDENACIÓN
# Ordenamos por fecha y estación, y reseteamos el índice
df_final = df_final.sort_values(by=['fecha', 'Estación']).reset_index(drop=True)

# 5. GUARDAR EL DATASET DEFINITIVO
nombre_final = "dataset_ia_clima_demanda.csv"
df_final.to_csv(nombre_final, index=False)

print("-" * 30)
print(f"🏆 ¡PROCESO COMPLETADO!")
print(f"📁 Archivo generado: {nombre_final}")
print(f"📈 Registros finales unidos: {len(df_final)}")
print(f"📅 Rango temporal: {df_final['fecha'].min().date()} al {df_final['fecha'].max().date()}")
print("-" * 30)

# Visualizar el resultado final
display(df_final.head(10))

📖 Cargando archivos...


/tmp/ipykernel_3120/4285507770.py:5: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  df_clima_fest = pd.read_csv("clima_festivos_consolidado.csv")


📊 Registros clima: 3753816
📊 Registros demanda: 4741
------------------------------
🏆 ¡PROCESO COMPLETADO!
📁 Archivo generado: dataset_ia_clima_demanda.csv
📈 Registros finales unidos: 3753816
📅 Rango temporal: 2013-05-01 al 2026-03-31
------------------------------


,fecha,Estación,Provincia,Temperatura máxima (ºC),Temperatura mínima (ºC),Temperatura media (ºC),Racha (km/h),Velocidad máxima (km/h),Precipitación 00-24h (mm),Precipitación 00-06h (mm),Precipitación 06-12h (mm),Precipitación 12-18h (mm),Precipitación 18-24h (mm),nombre_festivo,es_festivo,dia_semana,tipo_dia,demanda_mw
0,2013-05-01,A Cañiza,Pontevedra,15.2,10.3,12.8,NaN,NaN,19.0,10.4,3.8,0.8,4.0,Labor Day,1,2,Festivo,588773.328
1,2013-05-01,A Coruña,A Coruña,19.7,15.0,17.4,66.0,31.0,1.2,1.0,0.0,0.2,0.0,Labor Day,1,2,Festivo,588773.328
2,2013-05-01,A Coruña Aeropuerto,A Coruña,19.2,14.8,17.0,68.0,39.0,0.1,0.0,0.0,0.1,0.0,Labor Day,1,2,Festivo,588773.328
3,2013-05-01,A Estrada,Pontevedra,18.2,12.5,15.4,NaN,NaN,24.2,16.2,0.6,0.6,6.8,Labor Day,1,2,Festivo,588773.328
4,2013-05-01,A Gudiña,Ourense,15.2,8.4,11.8,NaN,NaN,5.4,0.8,2.0,0.0,2.6,Labor Day,1,2,Festivo,588773.328
5,2013-05-01,A Lama,Pontevedra,16.6,11.1,13.9,39.0,19.0,26.2,15.0,4.4,0.4,6.4,Labor Day,1,2,Festivo,588773.328
6,2013-05-01,A Pobra de Trives,Ourense,17.4,10.3,13.8,47.0,20.0,0.8,0.6,0.0,0.0,0.2,Labor Day,1,2,Festivo,588773.328
7,2013-05-01,Abadiño,Bizkaia,19.4,11.6,15.5,NaN,NaN,0.2,0.0,0.2,0.0,0.0,Labor Day,1,2,Festivo,588773.328
8,2013-05-01,Abanilla,Murcia,31.9,18.7,25.3,NaN,NaN,0.0,0.0,0.0,0.0,0.0,Labor Day,1,2,Festivo,588773.328
9,2013-05-01,Abenójar,Ciudad Real,25.5,13.4,19.4,25.0,15.0,0.0,0.0,0.0,0.0,0.0,Labor Day,1,2,Festivo,588773.328
